In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [2]:
retriever = WikipediaRetriever(top_k_results=2,lang='en')

In [3]:
query="the geopolitical history of india and pakistan from the perspective of chinese."

# docs = retriever.invoke(query)
# docs

## Vector Store Retriever

In [4]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv

In [5]:
load_dotenv('../.env')

True

In [6]:
documents = [
    Document (page_content="LangChain helps developers build LLM applications easily."),
    Document (page_content="Chroma is a vector database optimized for LLM-based search."),
    Document (page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [7]:
embedding_model=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [8]:
vector_store=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory='my_chroma_db',
    collection_name='retriever_collection_chroma'
)

In [9]:
retriever = vector_store.as_retriever(search_kwargs={'k':2})

In [10]:
query="what is chroma used for?"
retriever.invoke(query)

[Document(id='59cae74c-9d6e-43b1-8956-ee748fc4c279', metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.'),
 Document(id='863ebb9a-c1a4-443d-bcf8-cfc70ee73ff3', metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.')]

## MMR

this retriever helps to find out diverse thigns from vector store

In [11]:
docs = [
    Document (page_content="LangChain makes it easy to work with LLMs."),
    Document (page_content="LangChain is used to build LLM based applications."),
    Document (page_content="Chroma is used to store and search document embeddings."),
    Document (page_content="Embeddings are vector representations of text."),
    Document (page_content="MMR helps you get diverse results when doing similarity search."),
    Document (page_content="LangChain supports Chroma, FAISS, Pinecone, and more.")
]

In [12]:
from langchain_community.vectorstores import FAISS

In [13]:
embedding_model=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [14]:
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [15]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':3,'lambda_mult':1} #k is no. of results and lambda_mult= relevance-diversity balance
)

In [16]:
query="what is Langchain?"
retriever.invoke(query)

[Document(id='511b88a8-63ed-427a-9e09-4ecd17fb5b6d', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='7c6bba03-4dc0-4498-aafb-62e62e55e9d1', metadata={}, page_content='LangChain makes it easy to work with LLMs.'),
 Document(id='5c4c5748-fb1c-441d-a997-aa7e5697995f', metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.')]

In [17]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':3,'lambda_mult':0.5} #k is no. of results and lambda_mult= relevance-diversity balance
)

query="what is Langchain?"
retriever.invoke(query)

[Document(id='511b88a8-63ed-427a-9e09-4ecd17fb5b6d', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='cabda069-b50f-439a-a621-a93a7b7fcf47', metadata={}, page_content='MMR helps you get diverse results when doing similarity search.'),
 Document(id='5c4c5748-fb1c-441d-a997-aa7e5697995f', metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.')]

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import MultiQueryRetriever

In [19]:
all_docs = [
    Document (page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document (page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document (page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document (page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"})
]

In [20]:
embedding_model=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [21]:
vector_store = FAISS.from_documents(documents=all_docs, embedding=embedding_model) 

In [22]:
similarity_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':5} #k is no. of results and lambda_mult= relevance-diversity balance
)

In [23]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={'k':5}),
    llm=model
)

In [24]:
query ='How to improve energy levels and maintain balance?'
mul_res=multiquery_retriever.invoke(query)
sim_res=similarity_retriever.invoke(query)

In [25]:
print('multiquery_retriever\n\n')
for i in range(len(mul_res)):
    print(f'Result {i}\n',mul_res[i].page_content,'\n')

print('similarity_retriever\n\n')
for i in range(len(sim_res)):
    print(f'Result {i}\n',sim_res[i].page_content,'\n')

multiquery_retriever


Result 0
 Drinking sufficient water throughout the day helps maintain metabolism and energy. 

Result 1
 Consuming leafy greens and fruits helps detox the body and improve longevity. 

Result 2
 Regular walking boosts heart health and can reduce symptoms of depression. 

Result 3
 Mindfulness and controlled breathing lower cortisol and improve mental clarity. 

Result 4
 Deep sleep is crucial for cellular repair and emotional regulation. 

Result 5
 The solar energy system in modern homes helps balance electricity demand. 

similarity_retriever


Result 0
 Drinking sufficient water throughout the day helps maintain metabolism and energy. 

Result 1
 The solar energy system in modern homes helps balance electricity demand. 

Result 2
 Mindfulness and controlled breathing lower cortisol and improve mental clarity. 

Result 3
 Consuming leafy greens and fruits helps detox the body and improve longevity. 

Result 4
 Regular walking boosts heart health and can reduce 

## Contexual Compression Retriever

A Contextual Compression Retriever improves retrieval by first fetching relevant documents and then compressing them to only the most useful parts before sending them to the LLM. Instead of passing full documents, it uses an LLM or embedding-based filter to extract or keep only the sections that directly answer the query, reducing token usage, noise, and hallucinations while improving response quality in RAG pipelines.

In [37]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


In [38]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [39]:
embedding_model=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [40]:
vector_store = FAISS.from_documents(documents=docs,embedding=embedding_model)

In [41]:
base_retriever = vector_store.as_retriever(search_kwargs={'k':5})

In [42]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
compressor = LLMChainExtractor.from_llm(llm)

In [43]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [44]:
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [45]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis does not occur in animal cells.

--- Result 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
